# SGCL — Sensitivity-Guided Cross-Layer Soft-Error Resilience: full experimental pipeline

This notebook runs **every experiment** of the manuscript *A Unified Sensitivity Guided Cross Layer Framework for Soft Error Resilience in Machine Learning Systems* and regenerates all tables (3–8, 10, 11) and figures (2–8) from the raw per-seed outputs.

**Before you start:** `Runtime ▸ Change runtime type ▸ GPU` (T4 is enough). Every stage caches its result in Google Drive, so if the session disconnects you simply re-run the cells and the pipeline resumes where it stopped.

Budgets (`BUDGET` below):
* `fast` — N = 500 paired seeds kept, 1 000-image stratified test subset per trial, 20 training epochs, 3 fine-tuning epochs per pruning step, secondary sweep points with 100 seeds. ≈ 6–8 GPU-hours on a T4.
* `colab` — identical protocol/structure, 30 training epochs, 5 fine-tuning epochs per pruning step, N = 500 paired seeds on a fixed stratified 2 000-image test subset per trial (secondary λ/sparsity sweeps use 250 seeds). ≈ 8–12 GPU-hours on a T4 in total; can be split over several sessions.
* `paper` — the exact protocol of Table 2 (200 epochs, 30 fine-tuning epochs, full test split, N = 500 everywhere). ≈ 10× longer; use Colab Pro / a workstation GPU.

In [ ]:
BUDGET = "fast"           # "fast" | "colab" | "paper"
PAIRS  = "resnet18:cifar10,efficientnet_b0:cifar10,resnet18:gtsrb,efficientnet_b0:gtsrb"
EXTRA  = ""
LAM_OP   = 1e-7           # operating point (chosen from the baseline sweep: baseline CER of a few pp)
LAM_SENS = 1e-5           # lambda used only to estimate the layer-wise sensitivity ranking (activation-only, per block)

from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/sgcl_results_v3'     # NEW folder: never reuse a folder from an earlier code version
os.makedirs(WORK, exist_ok=True)
print('results will be stored in', WORK)

### 1. Get the code
Either upload `sgcl_pipeline.zip` to `/content/` (left sidebar ▸ Files ▸ Upload) or clone your GitHub repository.

In [ ]:
import os, zipfile, shutil
shutil.rmtree('/content/sgcl', ignore_errors=True)          # always start from the uploaded package, never from a patched copy
zipfile.ZipFile('/content/sgcl_pipeline.zip').extractall('/content/')
%cd /content/sgcl
!pip -q install -r requirements.txt
!python tests_smoke.py
!nvidia-smi --query-gpu=name,memory.total --format=csv

### 2. Run the whole pipeline
One cell, re-run it after every disconnection: finished stages print `[skip]` and the run resumes. Do not add any other cell that edits files in `/content/sgcl`.

In [ ]:
!python run_pipeline.py --budget $BUDGET --out $WORK/results --data /content/data --pairs $PAIRS \
    --lam-op $LAM_OP --lam-sens $LAM_SENS --critical-rule median --extra-archs "$EXTRA" --workers 2

### 3. Build all tables and figures from the raw outputs

In [ ]:
!python make_report.py --results $WORK/results --out $WORK/report_lam$LAM_OP --lam-op $LAM_OP
!ls $WORK/report_lam$LAM_OP/tables $WORK/report_lam$LAM_OP/figures
from IPython.display import Markdown, display, Image
import glob
for f in sorted(glob.glob(f'{WORK}/report_lam{LAM_OP}/tables/*.md')): display(Markdown(open(f).read()))
for f in sorted(glob.glob(f'{WORK}/report_lam{LAM_OP}/figures/*.png')): display(Image(f, width=420))

### 4. Package the results for me / Zenodo / GitHub
The first zip (small, a few MB) contains every JSON output and the report — **send me this one**. The second also contains the checkpoints (hundreds of MB) and is the one to deposit on Zenodo later.

In [ ]:
!cd $WORK && zip -qr sgcl_results_json.zip results report_lam$LAM_OP -x "results/ckpt/*" && ls -lh sgcl_results_json.zip
!cd $WORK && zip -qr sgcl_results_full.zip results report_lam$LAM_OP && ls -lh sgcl_results_full.zip
!cp -r /content/sgcl $WORK/code_snapshot 2>/dev/null; echo done